<a href="https://colab.research.google.com/github/MParvan/ecg-biometrics-bench/blob/main/experiments/Custom_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Injecting Custom PyTorch Architectures
The framework provides robust baselines (deepecg, resnet1d, transformer, etc.), but if you are researching novel neural network architectures, you will want to test your own model.

To integrate seamlessly with the biometric matching pipeline (which requires extracting embeddings for Gallery/Probe template matching), your custom model just needs to follow one simple rule: It must support returning feature embeddings when the classification head is disabled.

# Step 1: Write Your PyTorch Model
Your model should accept an include_top boolean argument. If False, it returns the high-dimensional feature vector. If True, it returns the Softmax logits.

## The contract your model must satisfy

The evaluation runners rely on exactly four properties. The
framework's own test suite checks these for every registered
architecture, so a model that satisfies them will work under all
eight protocols:

1. **Signature.** `__init__(self, in_channels, num_classes,
   include_top)`. The runners pass all three by keyword.
2. **Two output modes.** With `include_top=True` return logits of
   shape `(batch, num_classes)`; with `include_top=False` return a
   2-D embedding `(batch, embedding_dim)`.
3. **Length independence.** Beat length ranges from 76 samples
   (NSRDB at 128 Hz) to 600 (PTB at 1 kHz). Use
   `nn.AdaptiveAvgPool1d(1)` rather than a fixed `flatten`, or your
   model will only work on one dataset.
4. **Constant embedding width.** The embedding dimension must not
   depend on input length, or templates built on different datasets
   will not be comparable.

You can check your model against the same tests the built-in
architectures pass:

```bash
python -m pytest tests/test_literature_baselines.py -k ModelContract
```

In [ ]:
import torch
import torch.nn as nn

# 1. Define your custom architecture
class MyAwesomeECGNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=10, include_top=True):
        super(MyAwesomeECGNet, self).__init__()
        self.include_top = include_top

        # A simple Feature Extractor
        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1) # Flattens to (Batch, 64, 1)
        )

        # The Biometric Embedding Layer (e.g., 64-dimensional embedding)
        self.embedding_size = 64

        # The Classification Head (Only used for training or Task 1/3)
        self.classifier = nn.Linear(self.embedding_size, num_classes)

    def forward(self, x):
        # x shape: (Batch, Channels, Time)
        x = self.features(x)
        embeddings = x.view(x.size(0), -1) # Flatten to (Batch, 64)

        # Framework Routing Logic:
        if not self.include_top:
            # For Tasks 2, 4, 6, 8 (Template Verification matching)
            return embeddings

        # For Training and Tasks 1, 3 (Identification)
        logits = self.classifier(embeddings)
        return logits

# Step 2: Register Your Model in the Framework

Registration is a single dictionary entry. `main.py` defines
`MODEL_REGISTRY`, which backs both the `--model` command-line choices
and the model lookup, so adding one line makes your architecture
available to every evaluation protocol.

1. Paste your `MyAwesomeECGNet` class into `models.py`.
2. Import it in `main.py` and add one entry to `MODEL_REGISTRY`.

There is nothing else to change: the `--model` choices are generated
from the registry keys, so your tag appears in `--help` automatically.

In [ ]:
# In main.py

from models import (
    DeepECG, ResNet1D, RNN_ECG, HybridCNNLSTM, ECGTransformer,
    ECGXtractor, MobileNetGRU, MultiScaleCNN, SeparableResNet,
    MyAwesomeECGNet,          # <-- your architecture
)

MODEL_REGISTRY = {
    'deepecg': DeepECG,
    'resnet1d': ResNet1D,
    'rnn': RNN_ECG,
    'hybrid': HybridCNNLSTM,
    'transformer': ECGTransformer,
    'ecgxtractor': ECGXtractor,
    'mobilenet_gru': MobileNetGRU,
    'multiscale_cnn': MultiScaleCNN,
    'separable_resnet': SeparableResNet,
    'my_awesome_net': MyAwesomeECGNet,   # <-- one line, that is all
}

# Verify the framework can see it:
#     python main.py --help | grep my_awesome_net

# Step 3: Run Your Benchmarks
Now, your custom model is fully integrated. You can benchmark it using the CLI just like the default models!

In [ ]:
# Run the Ultimate Cross-Session Verification Task with your new model!
!python main.py \
  --dataset heartprint \
  --task 8 \
  --data_split_mode cross-session \
  --train_sessions session1 \
  --probe_sessions session2 \
  --model my_awesome_net \
  --epochs 100 \
  --save_results